In [ ]:
import json
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1 import make_axes_locatable
import arviz as az

from emu_renewal.inputs import get_world_shp
from emu_renewal.constants import (
    DATA_PATH,
    FULL_RUN,
    OXCGRT_LOCATION_CMAP,
    OXCGRT_COLMAP,
    MOB_LOCATION_NAME_MAP,
    OXCGRT_LOCS,
)
from emu_renewal.utils import get_analysis_paths

In [ ]:
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1, preserve_topology=True)

all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)
policy_codes = OXCGRT_COLMAP["custom"]

records = []
for iso3, analyses in analysis_paths.items():
    analysis_path = analyses["oxcgrt"]
    idata = az.from_netcdf(analysis_path / "idata_filtered.nc")
    medians = idata.posterior["ts_weights"].median(dim=("chain", "draw"))
    scale_floor = float(idata.posterior["scale_floor"].median(dim=("chain", "draw")))
    scale_exp = float(idata.posterior["scale_exp"].median(dim=("chain", "draw")))
    best_policy = int(medians.argmax().item())
    scale_factor = (1.0 - scale_floor) / float(np.sum(medians))
    row = {
        "ISO_A3": iso3,
        "best_policy": best_policy,
        "floor": scale_floor,
        "floor_complement": 1.0 - scale_floor,
        "scale_exp": scale_exp,
        "best_name": policy_codes[int(best_policy)]
    }
    for k, v in enumerate(medians):
        row[f"pol_{k}"] = float(v) * scale_factor
    records.append(row)
data_df = pd.DataFrame.from_records(records)

world = world.merge(data_df, on="ISO_A3", how="left")
missing = world[world["best_policy"].isna()]
exclude = world[(world["floor"] > 0.75) | (world["scale_exp"] < 0.75)]
mpl.rcParams["hatch.color"] = "lightgrey"

## Complement of the scale floor

In [ ]:
def plot_param_map(world, param_name, upper_val, excluded=None, title=""):
    fig, ax = plt.subplots(1, 1, figsize=(20, 10))
    world.boundary.plot(ax=ax, color="k", linewidth=0.4)
    ax.set_xticks([])
    ax.set_yticks([])
    missing = world[world["best_policy"].isna()]
    cax = make_axes_locatable(ax).append_axes("right", size=0.4, pad=0.25)
    world.plot(ax=ax, column=param_name, cmap="Blues", legend=True, vmin=0, vmax=upper_val, legend_kwds={"cax": cax})
    missing.plot(ax=ax, facecolor="white", edgecolor="none", hatch="//")
    ax.set_title(title, fontsize=22.0)
    if excluded is not None:
        excluded.plot(ax=ax, facecolor="lightgrey")

In [ ]:
plot_param_map(world, "floor_complement", 1.0)

## Scale exponent

In [ ]:
plot_param_map(world, "scale_exp", 2.0)

## Best policy

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 10))
world.boundary.plot(ax=ax, color="k", linewidth=0.4)
ax.set_xticks([])
ax.set_yticks([])

cmap = mcolors.ListedColormap(OXCGRT_LOCATION_CMAP)
avail = world[world["best_policy"].notna()]
avail.plot(ax=ax, column="best_name", edgecolor="none", cmap=cmap, vmin=0, vmax=len(policy_codes) - 1)
missing.plot(ax=ax, facecolor="white", edgecolor="none", hatch="//")
exclude.plot(ax=ax, facecolor="lightgrey")

handles = [Patch(facecolor=c, label=n) for n, c in zip(OXCGRT_LOCS.values(), OXCGRT_LOCATION_CMAP)]
ax.legend(handles=handles, loc="lower left", fontsize=16)

fig.savefig("best_policy.png")

In [ ]:
for a in range(len(medians)):
    policy = MOB_LOCATION_NAME_MAP[policy_codes[a]]
    plot_param_map(world, f"pol_{a}", 0.14, excluded=exclude, title=policy)
